# Автоматизация классификации растров для ArcGIS Pro

Блокнот предназначен для пакетной классификации растров по описательным файлам:
* .rmp — файл маппинга
* .csv — таблица атрибутов
* .clr — цветовая карта

В итоге получаешь классифицированный файл с типом byte с таблицей атрибутов (RAT), цветовой картой, с нормально названными каналами.
Можно собрать группу файлов в один многоканальный файл, но тогда слетает таблица атрибутов и цветовая шкала, т.к. ArcGis в многоканальных растрах эту красоту не поддерживает.

P.s. Скрипт не убирает за собой мусор. Не охота было прикручивать этот функционал.


In [1]:
import csv
import os
from pathlib import Path

# Импорт библиотек
import arcpy
from osgeo import gdal

# Разрешить перезапись выходных данных
arcpy.env.overwriteOutput = True

print("Библиотеки импортированы.")

Библиотеки импортированы.


## Пользовательская конфигурация

In [2]:
# Группа и пути к растрам, входящим в группу
# Итоговый растр будет называться по имени группы
# Если в группу входят несколько растров, то итоговый растр будет многоканальным, но без цветовой карты и RAT
INPUT_RASTERS_DICT: dict[str, list[Path]] = {
    # "Rx1DayAri40": [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb\Rx1DayAri40"),
    # ],
    # "Rx5DayAri40": [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\FloodFactors.gdb\Rx5DayAri40"),
    # ],
    # "slope_x100": [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\relief_analyze.tif\slope_x100"),
    # ],
    # "twi_x100": [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\relief_analyze.tif\twi_x100"),
    # ],    
    # "Flood": [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\flood_river_RP100y_depth.tif"),
    # ],
    # "HiHydroSoil" : [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\HiHydroSoil.tif")
    # ],
    # "WorldCover_infiltration" : [
    #     Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb\WorldCover_Mozaic")
    # ],
    "WorldCover_impedance" : [
        Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\GDBs\WorldCover.gdb\WorldCover_Mozaic")
    ],
    # Добавьте другие группы по необходимости
}

# Путь к служебным файлам
# .rmp — файл маппинга, .csv — таблица атрибутов, .clr — цветовая карта
SERVICE_FILE_DIR = Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Class")

# Папка для выходных файлов
OUTPUT_DIR = Path(r"I:\docs\maxim\prj\gis\learn\FloodBenineSmail\Maps\Rasters\Factors")

# Папка для временных файлов
TEMP_DIR = Path(r"F:\temp\classify")

# Переопределение умолчаний для растров
# Словарь для переименования каналов
BAND_NAME_MAPPING: dict[str,str] = {
    # Ключ: имя исходного файла (stem), Значение: имя канала в итоговом растре
    # "WorldCover_Mozaic": "cover_infiltration",
    "WorldCover_Mozaic": "cover_impedance",
    "flood_river_RP100y_depth": "flood_class",
    "HiHydroSoil": "infiltration",
    # Если ключа нет в словаре, будет использовано дефолтное имя (например, исходное имя + `BAND_SUFFIX_DEFAULT`)
}

# Словарь для установки виртуального имени растра. Будет использоваться для поиска служебных файлов и создания временных
SERVICE_FILE_MAPPING: dict[str,Path] = {
    # Ключ: имя исходного файла (stem), Значение: виртуальное имя файла
    # "WorldCover_Mozaic": Path("WorldCover_infiltration"),
    "WorldCover_Mozaic": Path("WorldCover_impedance"),
}

# Словарь для установки имени столбца в таблице атрибутов растра (RAT)
COL_RAT_NAME_MAPPING: dict[str,str] = {
    # "WorldCover_Mozaic": "infiltr",
    "WorldCover_Mozaic": "imped",
    "HiHydroSoil": "infiltr",
}

# Умолчания
# Название столбца для описания классов
COL_RAT_FOR_CLASS_NAME = "Danger"
# Суффикс к названию канала растра
BAND_SUFFIX_DEFAULT = "_class"
# Атрибуты классификации
RAT_VALUES_DEFAULT = {
    1: "Допустимый",
    2: "Умеренный",
    3: "Существенный",
    4: "Экстремальный",
    5: "Катастрофический",
    6: "Запредельный"
}
# Цветовая карта
DEFAULT_CLR = """1 163 217 120
2 255 224 96
3 250 142 64
4 240 70 75
5 168 0 0
6 90 0 0
"""

# Создание директорий, если они не существуют
TEMP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Цветовая карта

## Вспомогательные функции

In [3]:
def try_load_rmp_file(rmp_path: Path) -> tuple[bool, list[str]] | None:
    """
    Читает файл RMP и проверяет наличие 'NODATA'.
    Возвращает кортеж (has_nodata, lines).
    """
    try:
        lines = [line.strip() for line in rmp_path.read_text().splitlines() if line.strip()]
        # Удаляем комментарии если есть (обычно в RMP их нет)
        lines = [line for line in lines if not line.startswith('#')]

        has_nodata = any(line.lower().startswith("nodata") for line in lines)
        return has_nodata, lines

    except FileNotFoundError:
        print(f"Ошибка: Файл преобразований не найден: {rmp_path}")
        return None

In [4]:
def try_as_number(element: str | None) -> int | float | str | None:
    if element is None:
        return None
    try:
        return int(element)
    except ValueError:
        try:
            return float(element)
        except ValueError:
            return element

In [5]:
def parse_rmp(lines) -> arcpy.sa.RemapRange | str:
    """
    Преобразует строки RMP в таблицу перекодировки для arcpy.ddd.Reclassify.
    Пытается адаптировать форматы 'min max : val' или 'old : new'.
    """
    # Заменяем двоеточие на пробел для унификации разделителей
    remap_parts = [line.replace(':', ' ') for line in lines]
    split_parts: list[list] = [part.split() for part in remap_parts]
    correct_NoData = [["NoDATA" if elem.lower() == "nodata" else elem for elem in part] for part in split_parts]
    
    # Что-то не срабатывает RemapRange с NoData...
    # complete_parts: list[list] = [part if len(part) == 3 else [part[0], *part] for part in correct_NoData]
    # number_parts = [[try_as_number(elem) for elem in part] for part in complete_parts]
    # return arcpy.sa.RemapRange(number_parts)

    str_r = ';'.join([' '.join([str(e) for e in elem]) for elem in correct_NoData])
    

    return str_r
    

In [6]:
def rename_raster_bands(raster_path: Path, new_band_names: list[str]):
    """
    Переименовывает каналы растра, используя гибридный подход (arcpy + gdal).
    """
    print(f"      Переименование каналов для {raster_path.name}...")

    rast_obj = arcpy.Raster(str(raster_path))

    # Открываем через GDAL (1 = Update)
    ds = gdal.Open(str(raster_path), 1)

    if ds is None:
        raise RuntimeError(f"      Ошибка GDAL: Не могу открыть файл {raster_path.name}. Возможно, он заблокирован ArcGIS.")

    for i, new_name in enumerate(new_band_names, start=1):
        # 1. Обновление через ArcPy
        if new_name not in rast_obj.bandNames:
            rast_obj.renameBand(i, new_name)
        else:
            print(f"      Канал с именем '{new_name}' уже существует, arcPy это не одобряет")

        # 2. Обновление через GDAL
        band = ds.GetRasterBand(i)
        if band:
            band.SetDescription(new_name)
            band.SetMetadataItem("BandName", new_name)

    # Сброс и освобождение
    band = None
    ds = None
    rast_obj = None
    print("      Каналы успешно переименованы.")

In [7]:
def add_raster_attributes(raster_path: Path, original_raster_path: Path, csv_path: Path | None = None):
    """
    Создает и заполняет атрибутивную таблицу (RAT).
    """
    print(f"      Добавление атрибутов для {raster_path}...")
    try:
        arcpy.management.BuildRasterAttributeTable(str(raster_path), "Overwrite")

        value_map = RAT_VALUES_DEFAULT
        # Чтение CSV если есть
        if csv_path and csv_path.exists():
            print(f"      Используется CSV: {csv_path.name}")
            custom_map = {}
            try:
                with open(csv_path, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    for row in reader:
                        if len(row) >= 2 and row[0].strip().isdigit():
                            custom_map[int(row[0].strip())] = row[1].strip()
                if custom_map:
                    value_map = custom_map
            except Exception as e:
                print(f"      Ошибка чтения CSV: {e}. Используются значения по умолчанию.")

        # Добавление поля и обновление значений        
        # Проверяем, существует ли поле, если нет - создаем
        filed_name = COL_RAT_NAME_MAPPING.get(original_raster_path.stem, COL_RAT_FOR_CLASS_NAME)
        fields = [f.name for f in arcpy.ListFields(str(raster_path))]
        if filed_name not in fields:
            arcpy.management.AddField(str(raster_path), filed_name, "TEXT", field_length=100)


        # Обновление строк
        with arcpy.da.UpdateCursor(str(raster_path), ["Value", filed_name]) as cursor:
            for row in cursor:
                val = row[0]
                if val in value_map:
                    row[1] = value_map[int(val)]
                    cursor.updateRow(row)
        print("      RAT обновлена.")

    except Exception as e:
        print(f"      Ошибка при работе с RAT: {e}")

In [8]:
def apply_colormap(raster_path: Path, clr_path: Path | None = None):
    """
    Применяет цветовую карту.
    """
    print(f"      Применение цветовой карты для {raster_path}...")
    try:
        target_clr = clr_path
        temp_clr = None

        if not (clr_path and clr_path.exists()):
            # Создаем временный файл с дефолтными значениями
            print("      Используется цветовая карта по умолчанию.")

            temp_clr = raster_path.with_suffix(".clr")
            temp_clr.write_text(DEFAULT_CLR)
            target_clr = temp_clr
        else:
            print(f"      Используется CLR файл: {clr_path.name}")

        arcpy.management.AddColormap(str(raster_path), input_CLR_file=str(target_clr))

        print("      Цветовая карта применена.")
    except Exception as e:
        print(f"      Ошибка применения цветовой карты. Обрати внимание, что тип пикселя д.б. без знаковым, например, 8_BIT_UNSIGNED")
        print(f"      {e}")
    finally:
        if temp_clr and temp_clr.exists():
            temp_clr.unlink()
        

In [9]:
def process_single_raster(
        raster_path: Path,
        service_files_dir: Path,
        temp_dir: Path
) -> Path | None:
    """
    Выполняет шаги: Статистика, Классификация (по .rmp), Конвертация в 8 бит, цветовая карта и таблица атрибутов растра.
    Возвращает путь к обработанному временному растру или None при ошибке.
    """
    
    if not raster_path.exists() and not raster_path.parent.exists():
        print(f"ПРЕДУПРЕЖДЕНИЕ: Растр {raster_path} не найден. Пропуск.")
        return None

    print(f"\n-> Обработка растра: {raster_path.name}\n")

    # 1. Вычисление статистики
    print("   1. Вычисление статистики...")
    arcpy.management.CalculateStatistics(str(raster_path), skip_existing="SKIP_EXISTING")

    # Пути для дальнейшей обработки
    virtual_stem = SERVICE_FILE_MAPPING.get(raster_path.stem, raster_path).stem
    rmp_file = service_files_dir / Path(virtual_stem).with_suffix(".rmp")
    classified_raster_temp = temp_dir / (virtual_stem + "_classified.tif")
    final_single_raster = temp_dir / (virtual_stem + "_8bit.tif")
    csv_file = service_files_dir / (virtual_stem + ".csv")
    clr_file = service_files_dir / (virtual_stem + ".clr")

    # 2. Поиск файла преобразований (.rmp)
    if not rmp_file.exists():
        print(f"   ОШИБКА: Файл преобразований не найден: {rmp_file}")
        return None

    # 2.1 Чтение содержимого
    if not (rmp := try_load_rmp_file(rmp_file)):
        return None
    has_nodata, rmp_lines = rmp


    # 2.2 Классификация
    try:
        if not has_nodata:
            print("   2. Классификация (ReclassByASCIIFile)...")
            arcpy.ddd.ReclassByASCIIFile(
                str(raster_path), str(rmp_file), str(classified_raster_temp), "NODATA"
            )
        else:
            print("   2. Классификация (Reclassify, обнаружен NoData)...")
            remap = parse_rmp(rmp_lines)
            arcpy.ddd.Reclassify(
                str(raster_path), "Value", remap, str(classified_raster_temp), "NODATA"
            )
    except Exception as e:
        print(f"   ОШИБКА при классификации: {e}")
        return None

    # 3. Изменение глубины на 8 бит
    # 8_BIT_SIGNED нельзя использовать, т.к. не применится цветовая карта
    print("   3. Конвертация в 8_BIT_UNSIGNED ...")

    try:
        arcpy.management.CopyRaster(
            str(classified_raster_temp), str(final_single_raster), pixel_type="8_BIT_UNSIGNED"
        )
    except Exception as e:
        print(f"      ОШИБКА при конвертации: {e}")
        return None

    # 4-5. Добавляем таблицу атрибутов и цветовую карту
    print("   5. Добавляем цветовую карту...")
    apply_colormap(final_single_raster, clr_file)
    print("   4. Добавляем таблицу атрибутов...")
    add_raster_attributes(final_single_raster, raster_path, csv_file)

    return final_single_raster

In [10]:
def create_composite_raster(
        processed_rasters: list[Path],
        output_path: Path
) -> None:
    """
    Выполняет шаг 4: Объединение растров или копирование единственного.
    """
    if len(processed_rasters) > 1:
        print(f"   6. Объединение {len(processed_rasters)} растров в {output_path.name}...")
        inputs_str = ";".join([str(p) for p in processed_rasters])
        arcpy.management.CompositeBands(inputs_str, str(output_path))
    else:
        print(f"   6. Копирование единственного растра в {output_path.name}...")
        arcpy.management.CopyRaster(str(processed_rasters[0]), str(output_path))

In [11]:
def finalize_raster_bands(
        final_raster: Path,
        original_names: list[str],
        service_files_dir: Path,
        name_mapping: dict[str, str]
) -> None:
    """
    Переименование каналов (по словарю)
    """
    print("   7. Переименование каналов...")

    # Генерация новых имен на основе словаря или дефолтного шаблона
    new_band_names = []
    for orig_name in original_names:
        # Ищем в словаре, если нет — используем имя по умаолчанию
        new_name = name_mapping.get(orig_name, f"{orig_name}{BAND_SUFFIX_DEFAULT}")
        new_band_names.append(new_name)

    # Вызов внешней функции переименования
    rename_raster_bands(final_raster, new_band_names)

In [12]:
# --- ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ (Refactored) ---

def main_processing_loop(input_rasters_dict, output_dir, service_files_dir, temp_dir, band_name_mapping):

    for group_name, raster_list in input_rasters_dict.items():
        print(f"\n{'=' * 20}\nОбработка группы: {group_name}\n{'=' * 20}\n")

        processed_rasters = []
        original_names = []
        group_failed = False

        # --- Шаг 1-5: Обработка каждого растра по отдельности ---
        for raster_path in raster_list:
            result_path = process_single_raster(raster_path, service_files_dir, temp_dir)

            if result_path is None:
                print("   Работа с данной группой остановлена из-за ошибки в растре.")
                group_failed = True
                break

            processed_rasters.append(result_path)
            original_names.append(raster_path.stem)

        if group_failed or not processed_rasters:
            print(f"Группа {group_name} пропущена.\n")
            continue

        # --- Шаг 6: Сборка многоканального растра ---
        final_output_raster = output_dir / (group_name + ".tif")
        try:
            create_composite_raster(processed_rasters, final_output_raster)
        except Exception as e:
            print(f"Критическая ошибка при сборке растра {group_name}: {e}")
            continue

        # --- Шаг 7: Пост-обработка многоканального растра (имена каналов) ---
        finalize_raster_bands(
            final_output_raster,
            original_names,
            service_files_dir,
            band_name_mapping
        )

        print(f"\nГруппа {group_name} успешно обработана.")

    print("Все задачи выполнены.")

## Запуск 

In [13]:
arcpy.env.overwriteOutput = True
main_processing_loop(INPUT_RASTERS_DICT, OUTPUT_DIR, SERVICE_FILE_DIR, TEMP_DIR, BAND_NAME_MAPPING)


Обработка группы: WorldCover_impedance


-> Обработка растра: WorldCover_Mozaic

   1. Вычисление статистики...
   2. Классификация (ReclassByASCIIFile)...
   3. Конвертация в 8_BIT_SIGNED ...
   5. Добавляем цветовую карту...
      Применение цветовой карты для F:\temp\classify\WorldCover_impedance_8bit.tif...
      Используется цветовая карта по умолчанию.
      Цветовая карта применена.
   4. Добавляем таблицу атрибутов...
      Добавление атрибутов для F:\temp\classify\WorldCover_impedance_8bit.tif...
      Используется CSV: WorldCover_impedance.csv
      RAT обновлена.
   6. Копирование единственного растра в WorldCover_impedance.tif...
   7. Переименование каналов...
      Переименование каналов для WorldCover_impedance.tif...
      Каналы успешно переименованы.

Группа WorldCover_impedance успешно обработана.
Все задачи выполнены.


C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\site-packages\osgeo\gdal.py:315: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(
